# PhysioNet EEG Motor Imagery — Analysis Notebook

This notebook demonstrates:
1. Loading the PhysioNet EEG Motor Imagery dataset
2. Visualising raw EEG signals
3. Visualising ERPs (Event-Related Potentials)
4. Extracting features and plotting their distributions
5. Training a fine-tuned model
6. Comparing simulated vs real EEG accuracy
7. Plotting a confusion matrix for real EEG data

In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print('Setup complete.')

## 1. Load PhysioNet Dataset

In [ ]:
from datasets.physionet_loader import download_physionet, load_epochs, extract_physionet_features

# Download subjects 1-3 (skip if already downloaded)
try:
    download_physionet(subject_ids=[1, 2, 3], data_dir='../datasets/physionet/')
    print('Download complete.')
except Exception as e:
    print(f'Download note: {e}')

In [ ]:
# Load epochs for subject 1
epochs = load_epochs(subject_id=1, data_dir='../datasets/physionet/')
print(epochs)

## 2. Visualise Raw EEG Signals

In [ ]:
import mne

# Plot raw EEG (first epoch)
data = epochs.get_data(units='uV')   # (n_epochs, n_channels, n_times)
times = epochs.times
ch_names = epochs.ch_names[:8]       # show first 8 channels

fig, axes = plt.subplots(8, 1, figsize=(14, 10), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(times, data[0, i, :], linewidth=0.8, color=f'C{i}')
    ax.set_ylabel(ch_names[i], fontsize=8)
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw EEG — Subject 1, Epoch 0 (8 channels)', y=1.01)
plt.tight_layout()
plt.show()

## 3. Visualise ERPs (Event-Related Potentials)

In [ ]:
# Compute ERP (mean across trials for each event type)
event_labels = {v: k for k, v in epochs.event_id.items()}
event_codes = epochs.events[:, 2]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (code, label) in zip(axes, epochs.event_id.items()):
    mask = event_codes == epochs.event_id[code]
    erp = data[mask, :8, :].mean(axis=0)   # average over matching epochs
    for ch in range(8):
        ax.plot(times, erp[ch], alpha=0.7, linewidth=0.9, label=ch_names[ch])
    ax.set_title(f'ERP — {label}', fontsize=10)
    ax.set_xlabel('Time (s)')
    ax.axvline(0, color='k', linewidth=0.8, linestyle='--')

axes[0].set_ylabel('Amplitude (µV)')
axes[0].legend(fontsize=7, loc='upper right')
plt.suptitle('Event-Related Potentials (ERP) — Subject 1', y=1.02)
plt.tight_layout()
plt.show()

## 4. Extract Features and Plot Distributions

In [ ]:
X, y = extract_physionet_features(epochs)
print(f'Features shape: {X.shape}  Labels shape: {y.shape}')
print(f'Classes: {dict(zip(*np.unique(y, return_counts=True)))}')

# Plot feature distributions for 3 classes
class_names = ['Rest (T0)', 'Left Hand (T1)', 'Right Hand (T2)']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for cls in range(3):
    ax = axes[cls]
    mask = y == cls
    if mask.sum() == 0:
        continue
    ax.hist(X[mask, :8].flatten(), bins=40, alpha=0.7, color=f'C{cls}')
    ax.set_title(class_names[cls])
    ax.set_xlabel('Feature value')
    ax.set_ylabel('Count')

plt.suptitle('Feature Distributions by Class', y=1.02)
plt.tight_layout()
plt.show()

## 5. Train Fine-tuned Model

In [ ]:
from model.finetune import finetune_on_physionet

finetune_on_physionet(subjects=[1, 2, 3], data_dir='../datasets/physionet/')
print('Fine-tuning complete.')

## 6. Compare Simulated vs Real EEG Accuracy

In [ ]:
from data.simulate_eeg import generate_eeg_data
from preprocessing.filter import extract_features
from tensorflow import keras
from model.train import MODEL_PATH

# Load base model
base_model = keras.models.load_model(str(MODEL_PATH))

# Evaluate on simulated data
sim_signals, sim_labels = generate_eeg_data(n_samples=200)
sim_features, _ = extract_features(sim_signals, normalize=True)
_, sim_acc = base_model.evaluate(sim_features, sim_labels, verbose=0)

# Evaluate on real PhysioNet data (binary: rest vs. movement)
y_binary = (y > 0).astype(int)   # 0=rest, 1=movement
_, real_acc = base_model.evaluate(X, y_binary, verbose=0)

print(f'Simulated EEG accuracy : {sim_acc * 100:.1f}%')
print(f'Real PhysioNet accuracy: {real_acc * 100:.1f}%')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Simulated EEG', 'Real PhysioNet'],
              [sim_acc * 100, real_acc * 100],
              color=['#2196f3', '#4caf50'])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title('Simulated vs Real EEG Classification Accuracy')
for bar, val in zip(bars, [sim_acc, real_acc]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f'{val * 100:.1f}%',
            ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Confusion Matrix for Real EEG Data

In [ ]:
from tensorflow import keras
from model.finetune import FINETUNED_MODEL_PATH
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

ft_model = keras.models.load_model(str(FINETUNED_MODEL_PATH))
y_pred = ft_model.predict(X, verbose=0).argmax(axis=1)

cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Rest', 'Left Hand', 'Right Hand'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Fine-tuned Model on Real EEG')
plt.tight_layout()
plt.show()